# Lab 2 Phase 4B: Depthwise-Separable Dual Attention Network (DSDAN)

Same-resolution image restoration (256x256 -> 256x256) using a novel architecture
with depthwise-separable convolutions for efficient wide feature extraction (128ch),
combined with dual attention: SE (channel) + SpatialGate (spatial).

**Architecture:** Stem -> 12 x DSDABlock (DW-Sep + SE + SpatialGate) -> Tail + global skip.

**NPU-native:** Conv2d, DepthwiseConv, BatchNorm2d, PReLU, GlobalAvgPool, HardSigmoid
**CPU fallback:** Adding (13), Multiply (24)

**Kernel:** Select `Python 3.9 (torch)` or an equivalent PyTorch notebook image before running.

**Modal notebook setup:** attach the dataset volume at `/mnt/data` and a writable output volume at `/mnt/runs`.
You can override paths with `LAB2_DATA_ROOT` and `LAB2_OUTPUT_DIR` if your mounts differ.

In [ ]:
from collections import defaultdict
from pathlib import Path
from typing import Optional
import copy
import hashlib
import io
import json
import math
import os
import random
import time
import warnings

warnings.filterwarnings("ignore", message=".*legacy TorchScript-based ONNX.*")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from PIL import Image, ImageFilter, ImageOps
from torchvision import transforms

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

## 1. Data Loading

Hybrid data pipeline from Phase 3: original paired SR data augmented with ImageNet-backed
synthetic same-resolution restoration samples.

- Original train: `Data/HR_train*` / `Data/LR_train*` (3036 pairs)
- Original val: `Data/val/HR_val` / `Data/val/LR_val` (100 pairs)
- ImageNet train: `Data/ImageNet/imagenet_train20a` (up to 3000 samples)
- ImageNet val: `Data/ImageNet/imagenet_val20` (up to 300 samples)

Synthetic LR is generated from HR via blur, downsample-upsample, and JPEG degradation.

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()


def env_flag(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


def env_int(name: str, default: int) -> int:
    value = os.getenv(name)
    return int(value) if value is not None else default


def env_float(name: str, default: float) -> float:
    value = os.getenv(name)
    return float(value) if value is not None else default


def has_expected_layout(root: Path) -> bool:
    required = [
        root / "HR_train",
        root / "LR_train",
        root / "val" / "HR_val",
        root / "val" / "LR_val",
    ]
    imagenet_roots = [root / "ImageNet", root / "course_files_export"]
    imagenet_ok = any(
        (image_root / "imagenet_train20.txt").exists() and (image_root / "imagenet_val20.txt").exists()
        for image_root in imagenet_roots
    )
    return all(path.exists() for path in required) and imagenet_ok


def resolve_data_root() -> Path:
    candidates = []
    env_root = os.getenv("LAB2_DATA_ROOT")
    if env_root:
        base = Path(env_root).expanduser()
        candidates.extend([base, base / "Data"])
    candidates.extend([
        NOTEBOOK_DIR / "Data",
        NOTEBOOK_DIR.parent / "Data",
        Path("/mnt/data"),
        Path("/mnt/data/Data"),
    ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if has_expected_layout(candidate):
            return candidate.resolve()

    searched = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not locate the dataset. Set LAB2_DATA_ROOT or attach the Modal volume at /mnt/data.\n"
        f"Searched:\n{searched}"
    )


def resolve_output_dir() -> Path:
    env_dir = os.getenv("LAB2_OUTPUT_DIR")
    if env_dir:
        return Path(env_dir).expanduser().resolve()

    for base in [Path("/mnt/runs"), Path("/mnt/output"), NOTEBOOK_DIR / "runs"]:
        if base.exists():
            return (base / "phase4b_dsdan").resolve()
    return (NOTEBOOK_DIR / "runs" / "phase4b_dsdan").resolve()


def resolve_image_root(data_root: Path) -> Path:
    for candidate in [data_root / "ImageNet", data_root / "course_files_export"]:
        if (candidate / "imagenet_train20.txt").exists() and (candidate / "imagenet_val20.txt").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate ImageNet manifest under {data_root}")


DATA_ROOT = resolve_data_root()
PROJECT_ROOT = DATA_ROOT.parent if DATA_ROOT.name == "Data" else DATA_ROOT
IMAGE_ROOT = resolve_image_root(DATA_ROOT)
OUTPUT_DIR = resolve_output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IS_MODAL_RUNTIME = str(DATA_ROOT).startswith("/mnt/") or str(OUTPUT_DIR).startswith("/mnt/")

HR_TRAIN_ROOT = DATA_ROOT / "HR_train"
LR_TRAIN_ROOT = DATA_ROOT / "LR_train"
HR_VAL_DIR = DATA_ROOT / "val" / "HR_val"
LR_VAL_DIR = DATA_ROOT / "val" / "LR_val"
IMAGENET_TRAIN_ROOT = IMAGE_ROOT / "imagenet_train20a"
IMAGENET_VAL_ROOT = IMAGE_ROOT / "imagenet_val20"
IMAGENET_TRAIN_LIST = IMAGE_ROOT / "imagenet_train20.txt"
IMAGENET_VAL_LIST = IMAGE_ROOT / "imagenet_val20.txt"

for required in [
    HR_TRAIN_ROOT, LR_TRAIN_ROOT, HR_VAL_DIR, LR_VAL_DIR,
    IMAGENET_TRAIN_ROOT, IMAGENET_VAL_ROOT, IMAGENET_TRAIN_LIST, IMAGENET_VAL_LIST,
]:
    assert required.exists(), f"Missing required path: {required}"

print(f"Runtime:      {'modal' if IS_MODAL_RUNTIME else 'local'}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")


def collect_paired_by_subfolder(lr_root: Path, hr_root: Path) -> list[tuple[Path, Path, str]]:
    hr_subdirs = sorted([d for d in hr_root.iterdir() if d.is_dir()])
    pairs = []
    for hr_sub in hr_subdirs:
        suffix = hr_sub.name.replace("HR_train", "")
        lr_sub = lr_root / f"LR_train{suffix}"
        if not lr_sub.exists():
            print(f"  WARNING: no matching LR subfolder for {hr_sub.name}")
            continue
        hr_imgs = {p.stem: p for p in sorted(hr_sub.glob("*.png"))}
        lr_imgs = {p.stem: p for p in sorted(lr_sub.glob("*.png"))}
        for stem in sorted(set(hr_imgs) & set(lr_imgs)):
            pairs.append((lr_imgs[stem], hr_imgs[stem], f"{hr_sub.name}/{stem}"))
    return pairs


def collect_paired_flat(lr_dir: Path, hr_dir: Path) -> list[tuple[Path, Path, str]]:
    hr_imgs = {p.stem: p for p in sorted(hr_dir.glob("*.png"))}
    lr_imgs = {p.stem: p for p in sorted(lr_dir.glob("*.png"))}
    common = sorted(set(hr_imgs) & set(lr_imgs))
    return [(lr_imgs[s], hr_imgs[s], s) for s in common]


def read_imagenet_manifest(path: Path) -> list[tuple[str, int]]:
    rows = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 2:
            continue
        rows.append((parts[0], int(parts[1])))
    return rows


def collect_imagenet_records(rows: list[tuple[str, int]], root: Path, split: str) -> list[dict]:
    records, missing = [], 0
    for filename, class_id in rows:
        synset = filename.split("_")[0]
        path = (root / synset / filename) if split == "train" else (root / filename)
        if not path.exists():
            missing += 1
            continue
        records.append({"path": path, "stem": path.stem, "class_id": class_id, "split": split, "synset": synset})
    if missing:
        print(f"WARNING: skipped {missing} missing ImageNet {split} files")
    return records


train_pairs = collect_paired_by_subfolder(LR_TRAIN_ROOT, HR_TRAIN_ROOT)
val_pairs = collect_paired_flat(LR_VAL_DIR, HR_VAL_DIR)
imagenet_train_rows = read_imagenet_manifest(IMAGENET_TRAIN_LIST)
imagenet_val_rows = read_imagenet_manifest(IMAGENET_VAL_LIST)
imagenet_train_records = collect_imagenet_records(imagenet_train_rows, IMAGENET_TRAIN_ROOT, split="train")
imagenet_val_records = collect_imagenet_records(imagenet_val_rows, IMAGENET_VAL_ROOT, split="val")

print(f"Original paired train: {len(train_pairs)}")
print(f"Original paired val:   {len(val_pairs)}")
print(f"ImageNet train:        {len(imagenet_train_records)}")
print(f"ImageNet val:          {len(imagenet_val_records)}")

In [ ]:
DATA_CFG = {
    "train_patch_size": 224,
    "eval_size": 256,
    "random_scale_pad": 32,
    "cutout_prob": 0.35,
    "cutout_ratio": 0.18,
    "lr_noise_prob": 0.30,
    "lr_noise_std": 0.015,
    "lr_blur_prob": 0.70,
    "jpeg_prob": 0.50,
    "jpeg_quality_min": 40,
    "jpeg_quality_max": 90,
    "downsample_scales": (2, 3, 4),
    "imagenet_train_limit": 3000,
    "imagenet_val_limit": 300,
    "train_eval_subset_size": 128,
}
BATCH_SIZE = env_int("LAB2_BATCH_SIZE", 12 if device.type == "cuda" else 8)
NUM_WORKERS = env_int("LAB2_NUM_WORKERS", min(4, os.cpu_count() or 0) if device.type == "cuda" else 0)
SEED = 255
RUN_SMOKE_TEST = env_flag("LAB2_RUN_SMOKE_TEST", True)
TO_TENSOR = transforms.ToTensor()
LOADER_KWARGS = {"num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda"}


def seeded_rng(key: str) -> random.Random:
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return random.Random(int(digest[:16], 16))


def take_manifest_subset(records, limit, seed):
    if limit is None or limit >= len(records):
        return list(records)
    rng = random.Random(seed)
    items = list(records)
    rng.shuffle(items)
    return items[:limit]


def random_crop_pair(lr_img, hr_img, size, rng):
    if lr_img.width == size and lr_img.height == size:
        return lr_img, hr_img
    x0 = rng.randint(0, lr_img.width - size)
    y0 = rng.randint(0, lr_img.height - size)
    box = (x0, y0, x0 + size, y0 + size)
    return lr_img.crop(box), hr_img.crop(box)


def random_crop_single(img, size, rng):
    if img.width == size and img.height == size:
        return img
    x0 = rng.randint(0, img.width - size)
    y0 = rng.randint(0, img.height - size)
    return img.crop((x0, y0, x0 + size, y0 + size))


def augment_pair(lr_img, hr_img, rng):
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
        hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
        hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k > 0:
        rot = {1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k]
        lr_img = lr_img.transpose(rot)
        hr_img = hr_img.transpose(rot)
    return lr_img, hr_img


def augment_single(img, rng):
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k > 0:
        img = img.transpose({1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k])
    return img


def jpeg_roundtrip(img, quality):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def degrade_from_hr(hr_img, rng, cfg):
    lr_img = hr_img.copy()
    if rng.random() < cfg["lr_blur_prob"]:
        lr_img = lr_img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.2, 1.2)))
    scale = rng.choice(cfg["downsample_scales"])
    small = (max(1, hr_img.width // scale), max(1, hr_img.height // scale))
    lr_img = lr_img.resize(small, resample=BICUBIC).resize(hr_img.size, resample=BICUBIC)
    if rng.random() < cfg["jpeg_prob"]:
        lr_img = jpeg_roundtrip(lr_img, rng.randint(cfg["jpeg_quality_min"], cfg["jpeg_quality_max"]))
    return lr_img


def apply_tensor_regularization(lr_t, rng, cfg, train):
    if not train:
        return lr_t
    if rng.random() < cfg["lr_noise_prob"]:
        lr_t = (lr_t + torch.randn_like(lr_t) * cfg["lr_noise_std"]).clamp(0.0, 1.0)
    if rng.random() < cfg["cutout_prob"]:
        _, h, w = lr_t.shape
        cut = max(8, int(min(h, w) * cfg["cutout_ratio"]))
        x0 = rng.randint(0, w - cut)
        y0 = rng.randint(0, h - cut)
        fill = lr_t.mean().item()
        lr_t[:, y0:y0 + cut, x0:x0 + cut] = fill
    return lr_t


class PairedSRDataset(Dataset):
    def __init__(self, pairs, train, data_cfg, source_name):
        self.pairs = pairs
        self.train = train
        self.data_cfg = data_cfg
        self.source_name = source_name

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path, stem = self.pairs[idx]
        lr_img = Image.open(lr_path).convert("RGB")
        hr_img = Image.open(hr_path).convert("RGB")
        rng = random.Random(SEED + idx) if self.train else seeded_rng(f"{self.source_name}:{stem}")
        if self.train:
            lr_img, hr_img = random_crop_pair(lr_img, hr_img, self.data_cfg["train_patch_size"], rng)
            lr_img, hr_img = augment_pair(lr_img, hr_img, rng)
        else:
            lr_img = ImageOps.fit(lr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
            hr_img = ImageOps.fit(hr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
        lr_t = TO_TENSOR(lr_img)
        hr_t = TO_TENSOR(hr_img)
        lr_t = apply_tensor_regularization(lr_t, rng, self.data_cfg, train=self.train)
        return lr_t, hr_t, stem, self.source_name


class ImageNetSyntheticSRDataset(Dataset):
    def __init__(self, records, train, data_cfg, source_name):
        self.records = records
        self.train = train
        self.data_cfg = data_cfg
        self.source_name = source_name

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        hr_img = Image.open(record["path"]).convert("RGB")
        rng = random.Random(SEED + idx) if self.train else seeded_rng(f"{self.source_name}:{record['stem']}")
        if self.train:
            base_size = max(self.data_cfg["eval_size"], self.data_cfg["train_patch_size"] + self.data_cfg["random_scale_pad"])
            hr_img = ImageOps.fit(hr_img, (base_size, base_size), method=BICUBIC)
            hr_img = random_crop_single(hr_img, self.data_cfg["train_patch_size"], rng)
            hr_img = augment_single(hr_img, rng)
        else:
            hr_img = ImageOps.fit(hr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
        lr_img = degrade_from_hr(hr_img, rng, self.data_cfg)
        lr_t = TO_TENSOR(lr_img)
        hr_t = TO_TENSOR(hr_img)
        lr_t = apply_tensor_regularization(lr_t, rng, self.data_cfg, train=self.train)
        return lr_t, hr_t, record["stem"], self.source_name


def make_fixed_subset_loader(dataset, subset_size, batch_size, seed):
    subset_size = min(subset_size, len(dataset))
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)
    subset = Subset(dataset, indices[:subset_size])
    return DataLoader(subset, batch_size=batch_size, shuffle=False, **LOADER_KWARGS)


imagenet_train_used = take_manifest_subset(imagenet_train_records, DATA_CFG["imagenet_train_limit"], SEED)
imagenet_val_used = take_manifest_subset(imagenet_val_records, DATA_CFG["imagenet_val_limit"], SEED)

paired_train_dataset = PairedSRDataset(train_pairs, train=True, data_cfg=DATA_CFG, source_name="paired_train")
paired_train_eval_dataset = PairedSRDataset(train_pairs, train=False, data_cfg=DATA_CFG, source_name="paired_train")
paired_val_dataset = PairedSRDataset(val_pairs, train=False, data_cfg=DATA_CFG, source_name="paired_val")
imagenet_train_dataset = ImageNetSyntheticSRDataset(imagenet_train_used, train=True, data_cfg=DATA_CFG, source_name="imagenet_train")
imagenet_train_eval_dataset = ImageNetSyntheticSRDataset(imagenet_train_used, train=False, data_cfg=DATA_CFG, source_name="imagenet_train")
imagenet_val_dataset = ImageNetSyntheticSRDataset(imagenet_val_used, train=False, data_cfg=DATA_CFG, source_name="imagenet_val")

train_dataset = ConcatDataset([paired_train_dataset, imagenet_train_dataset])
train_eval_dataset = ConcatDataset([paired_train_eval_dataset, imagenet_train_eval_dataset])
val_dataset = ConcatDataset([paired_val_dataset, imagenet_val_dataset])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **LOADER_KWARGS)
train_eval_loader = make_fixed_subset_loader(train_eval_dataset, DATA_CFG["train_eval_subset_size"], BATCH_SIZE, SEED)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)
paired_val_loader = DataLoader(paired_val_dataset, batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)
imagenet_val_loader = DataLoader(imagenet_val_dataset, batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)

lr_batch, hr_batch, names, sources = next(iter(train_loader))
print(f"Batch size: {BATCH_SIZE} | Num workers: {NUM_WORKERS} | Smoke test: {RUN_SMOKE_TEST}")
print(f"Combined train: {len(train_dataset)} | paired: {len(paired_train_dataset)} | imagenet: {len(imagenet_train_dataset)}")
print(f"Combined val:   {len(val_dataset)} | paired: {len(paired_val_dataset)} | imagenet: {len(imagenet_val_dataset)}")
print(f"Train-eval subset: {len(train_eval_loader.dataset)}")
print(f"Batch shapes: LR={tuple(lr_batch.shape)}, HR={tuple(hr_batch.shape)}")

## 2. Model Definition: Depthwise-Separable Dual Attention Network (DSDAN)

Novel architecture using depthwise-separable convolutions for parameter-efficient
wide feature extraction (128 channels), combined with both channel attention (SE)
and spatial attention (depthwise gating).

**NPU-native ops:** Conv2d, DepthwiseConv, BatchNorm2d, PReLU, GlobalAvgPool, HardSigmoid
**CPU fallback:** Adding (skip connections), Multiply (SE + spatial gating)

In [ ]:
class StochasticDepth(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if not self.training or self.drop_prob <= 0.0:
            return x
        keep = 1.0 - self.drop_prob
        mask = keep + torch.rand((x.shape[0],) + (1,) * (x.ndim - 1), dtype=x.dtype, device=x.device)
        mask.floor_()
        return x * mask / keep


class SEBlock(nn.Module):
    """Channel attention: GlobalAvgPool -> Conv1x1 -> PReLU -> Conv1x1 -> HardSigmoid -> scale."""

    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, mid, 1, bias=True)
        self.act = nn.PReLU(num_parameters=mid)
        self.fc2 = nn.Conv2d(mid, channels, 1, bias=True)
        self.gate = nn.Hardsigmoid()

    def forward(self, x):
        w = self.gate(self.fc2(self.act(self.fc1(self.pool(x)))))
        return x * w


class SpatialGate(nn.Module):
    """Spatial attention via depthwise conv gating.
    DepthwiseConv7x7 (NPU native) -> BN (NPU native) -> HardSigmoid (NPU native) -> Multiply (CPU fallback).
    """

    def __init__(self, channels):
        super().__init__()
        self.dw = nn.Conv2d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.bn = nn.BatchNorm2d(channels)
        self.gate = nn.Hardsigmoid()

    def forward(self, x):
        return x * self.gate(self.bn(self.dw(x)))


class DSDABlock(nn.Module):
    """Depthwise-Separable Dual Attention block.
    DW3x3 -> BN -> PReLU -> PW1x1 -> BN -> PReLU -> DW3x3 -> BN -> PW1x1 -> BN
    -> SE (channel attn) -> SpatialGate (spatial attn) -> + skip.
    """

    def __init__(self, channels, reduction=4, dropout=0.0, drop_path=0.0):
        super().__init__()
        self.dw1 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.act1 = nn.PReLU(num_parameters=channels)
        self.pw1 = nn.Conv2d(channels, channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.act2 = nn.PReLU(num_parameters=channels)
        self.dw2 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.bn3 = nn.BatchNorm2d(channels)
        self.pw2 = nn.Conv2d(channels, channels, 1, bias=False)
        self.bn4 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction)
        self.sa = SpatialGate(channels)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0.0 else nn.Identity()
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, x):
        out = self.act1(self.bn1(self.dw1(x)))
        out = self.act2(self.bn2(self.pw1(out)))
        out = self.bn3(self.dw2(out))
        out = self.bn4(self.pw2(out))
        out = self.se(out)
        out = self.sa(out)
        out = self.drop_path(self.dropout(out))
        return x + out


class DSDANSR(nn.Module):
    """Depthwise-Separable Dual Attention Network for SR.
    12 blocks, 128 channels. Input/Output: [B, 3, H, W].
    """

    def __init__(self, num_blocks=12, channels=128, reduction=4, dropout=0.08, max_drop_path=0.08):
        super().__init__()
        drops = torch.linspace(0.0, max_drop_path, num_blocks).tolist()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(num_parameters=channels),
        )
        self.body = nn.Sequential(*[
            DSDABlock(channels, reduction, dropout, drops[i]) for i in range(num_blocks)
        ])
        self.tail = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, x):
        return x + self.tail(self.body(self.stem(x)))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


FORBIDDEN_TYPES = (nn.ReLU, nn.Sigmoid, nn.Softmax, nn.LayerNorm, nn.GroupNorm)

def assert_npu_compatible(model):
    for name, mod in model.named_modules():
        if isinstance(mod, FORBIDDEN_TYPES):
            raise TypeError(f"Forbidden NPU op '{name}': {mod.__class__.__name__}")


MODEL_CFG = {
    "num_blocks": 12,
    "channels": 128,
    "reduction": 4,
    "dropout": 0.08,
    "max_drop_path": 0.08,
}

model = DSDANSR(**MODEL_CFG).to(device)
assert_npu_compatible(model)

with torch.no_grad():
    dummy = torch.randn(1, 3, DATA_CFG["train_patch_size"], DATA_CFG["train_patch_size"], device=device)
    out = model(dummy)

print(f"Model: DSDANSR | Config: {MODEL_CFG}")
print(f"Parameters: {count_parameters(model):,}")
print(f"Input: {tuple(dummy.shape)} -> Output: {tuple(out.shape)}")

npu_ops = {"Conv": 0, "DWConv": 0, "BN": 0, "PReLU": 0, "GlobalAvgPool": 0, "HardSigmoid": 0}
cpu_fb = {"Adding": 0, "Multiply": 0}
for m in model.modules():
    if isinstance(m, nn.Conv2d):
        if m.groups == m.in_channels and m.in_channels > 1:
            npu_ops["DWConv"] += 1
        else:
            npu_ops["Conv"] += 1
    elif isinstance(m, nn.BatchNorm2d): npu_ops["BN"] += 1
    elif isinstance(m, nn.PReLU): npu_ops["PReLU"] += 1
    elif isinstance(m, nn.AdaptiveAvgPool2d): npu_ops["GlobalAvgPool"] += 1
    elif isinstance(m, nn.Hardsigmoid): npu_ops["HardSigmoid"] += 1
cpu_fb["Adding"] = MODEL_CFG["num_blocks"] + 1
cpu_fb["Multiply"] = MODEL_CFG["num_blocks"] * 2  # SE + SpatialGate per block
print(f"NPU-native ops: {npu_ops}")
print(f"CPU Fallback: {cpu_fb}")
print("Failed ops: 0")

## 3. Training Infrastructure

Phase 4 enhancements over Phase 3:
- **Combined loss**: 0.5 * Charbonnier + 0.5 * L1 for smooth yet sharp optimization
- **EMA (Exponential Moving Average)**: shadow model with decay=0.999 for stable validation and export
- **Gradient clipping**: max_norm=1.0 to prevent training instability in deeper networks
- Fair eval-mode train PSNR, periodic checkpoint archival, early stopping

In [ ]:
TRAIN_CFG = {
    "epochs": env_int("LAB2_EPOCHS", 80),
    "batch_size": BATCH_SIZE,
    "lr": env_float("LAB2_LR", 3e-4),
    "weight_decay": env_float("LAB2_WEIGHT_DECAY", 2e-4),
    "warmup_epochs": env_int("LAB2_WARMUP_EPOCHS", 5),
    "min_lr_ratio": env_float("LAB2_MIN_LR_RATIO", 0.05),
    "checkpoint_interval": env_int("LAB2_CHECKPOINT_INTERVAL", 10),
    "seed": SEED,
    "early_stop_patience": env_int("LAB2_EARLY_STOP_PATIENCE", 15),
    "grad_clip_norm": env_float("LAB2_GRAD_CLIP_NORM", 1.0),
    "ema_decay": env_float("LAB2_EMA_DECAY", 0.999),
    "charb_eps": env_float("LAB2_CHARB_EPS", 1e-6),
}
RESUME_TRAINING = env_flag("LAB2_RESUME_TRAINING", True)
RUN_DIAGNOSTICS = env_flag("LAB2_RUN_DIAGNOSTICS", True)
RUN_ONNX_EXPORT = env_flag("LAB2_RUN_ONNX_EXPORT", True)
RUN_CALIBRATION_EXPORT = env_flag("LAB2_RUN_CALIBRATION_EXPORT", True)

print(f"Config: {TRAIN_CFG}")
print(f"Resume: {RESUME_TRAINING}")
print(f"Diagnostics: {RUN_DIAGNOSTICS} | ONNX: {RUN_ONNX_EXPORT} | Calibration: {RUN_CALIBRATION_EXPORT}")


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def charbonnier_loss(pred, target, eps=1e-6):
    diff = pred - target
    return torch.mean(torch.sqrt(diff * diff + eps * eps))


def combined_loss(pred, target, eps=1e-6):
    return 0.5 * charbonnier_loss(pred, target, eps) + 0.5 * F.l1_loss(pred, target)


def compute_psnr(pred, target):
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


class EMA:
    """Exponential Moving Average of model parameters for smoother convergence."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model):
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup = {}


def lr_for_epoch(epoch, total, base_lr, warmup, min_ratio):
    if warmup > 0 and epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total - warmup - 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_ratio
    return min_lr + (base_lr - min_lr) * cosine


def train_one_epoch(model, loader, optimizer, cfg, ema=None):
    model.train()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for lr_img, hr_img, _, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(lr_img)
        loss = combined_loss(pred, hr_img, eps=cfg["charb_eps"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["grad_clip_norm"])
        optimizer.step()
        if ema is not None:
            ema.update(model)
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        with torch.no_grad():
            total_psnr += compute_psnr(pred.detach(), hr_img).sum().item()
        n += bs
    return {"train_loss": total_loss / max(1, n), "train_psnr_online": total_psnr / max(1, n)}


@torch.no_grad()
def evaluate_loader(model, loader, cfg, split_name):
    model.eval()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for lr_img, hr_img, _, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        pred = model(lr_img)
        loss = combined_loss(pred, hr_img, eps=cfg["charb_eps"])
        psnr = compute_psnr(pred, hr_img)
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        total_psnr += psnr.sum().item()
        n += bs
    return {f"{split_name}_loss": total_loss / max(1, n), f"{split_name}_psnr": total_psnr / max(1, n)}


def save_checkpoint(path, model, optimizer, epoch, metrics, best_psnr, best_epoch, ema=None):
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "best_psnr": best_psnr,
        "best_epoch": best_epoch,
        "model_cfg": MODEL_CFG,
        "train_cfg": TRAIN_CFG,
        "data_cfg": DATA_CFG,
    }
    if ema is not None:
        state["ema_shadow"] = ema.shadow
    torch.save(state, path)


def load_history(metrics_path):
    if not metrics_path.exists():
        return []
    return [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]


def fit(model, train_loader, train_eval_loader, val_loader, output_dir, cfg=TRAIN_CFG, resume=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.jsonl"
    last_ckpt_path = output_dir / "last.pt"
    best_ckpt_path = output_dir / "best.pt"

    optimizer = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    ema = EMA(model, decay=cfg["ema_decay"])
    start_epoch, best_psnr, best_epoch = 0, float("-inf"), -1
    history, cumulative_time = [], 0.0

    if resume and last_ckpt_path.exists():
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"]
        best_psnr = ckpt.get("best_psnr", float("-inf"))
        best_epoch = ckpt.get("best_epoch", -1)
        history = load_history(metrics_path)
        cumulative_time = sum(r.get("seconds", 0.0) for r in history)
        if "ema_shadow" in ckpt:
            ema.shadow = ckpt["ema_shadow"]
        else:
            ema = EMA(model, decay=cfg["ema_decay"])
        if history and best_epoch < 0:
            best_epoch = int(max(history, key=lambda r: r["val_psnr"])["epoch"])
        print(f"RESUMING from epoch {start_epoch}/{cfg['epochs']}")
        print(f"  Best PSNR so far: {best_psnr:.3f} dB (epoch {best_epoch})")
    else:
        set_seed(cfg["seed"])
        metrics_path.write_text("")
        print("FRESH START")

    if start_epoch >= cfg["epochs"]:
        print(f"Already trained {start_epoch} epochs. Increase epochs to continue.")
        return history

    print(f"\n{'epoch':>5} | {'lr':>8} | {'train_loss':>10} {'train_online':>12} {'train_eval':>11} | {'val_psnr':>8} | {'best':>8} | {'time':>5}")
    print("-" * 100)

    for epoch in range(start_epoch, cfg["epochs"]):
        epoch_lr = lr_for_epoch(epoch, cfg["epochs"], cfg["lr"], cfg["warmup_epochs"], cfg["min_lr_ratio"])
        for group in optimizer.param_groups:
            group["lr"] = epoch_lr

        t0 = time.time()
        train_metrics = train_one_epoch(model, train_loader, optimizer, cfg, ema=ema)

        ema.apply_shadow(model)
        train_eval_metrics = evaluate_loader(model, train_eval_loader, cfg, "train_eval")
        val_metrics = evaluate_loader(model, val_loader, cfg, "val")
        ema.restore(model)

        elapsed = time.time() - t0
        cumulative_time += elapsed

        row = {
            "epoch": epoch + 1, "lr": epoch_lr,
            "train_loss": train_metrics["train_loss"],
            "train_psnr_online": train_metrics["train_psnr_online"],
            "train_eval_psnr": train_eval_metrics["train_eval_psnr"],
            "val_loss": val_metrics["val_loss"],
            "val_psnr": val_metrics["val_psnr"],
            "seconds": round(elapsed, 1),
        }
        history.append(row)
        with metrics_path.open("a") as f:
            f.write(json.dumps(row) + "\n")

        is_best = row["val_psnr"] > best_psnr
        if is_best:
            best_psnr = row["val_psnr"]
            best_epoch = epoch + 1

        stale = (epoch + 1) - best_epoch
        should_stop = stale >= cfg["early_stop_patience"]

        save_checkpoint(last_ckpt_path, model, optimizer, epoch + 1, row, best_psnr, best_epoch, ema)
        if is_best:
            save_checkpoint(best_ckpt_path, model, optimizer, epoch + 1, row, best_psnr, best_epoch, ema)
        if (epoch + 1) % cfg["checkpoint_interval"] == 0 or should_stop:
            save_checkpoint(output_dir / f"epoch_{epoch+1:03d}.pt", model, optimizer, epoch + 1, row, best_psnr, best_epoch, ema)

        marker = " *" if is_best else ""
        print(
            f"{epoch+1:3d}/{cfg['epochs']:3d} | {epoch_lr:.2e} | "
            f"{row['train_loss']:10.6f} {row['train_psnr_online']:10.3f} {row['train_eval_psnr']:10.3f} | "
            f"{row['val_psnr']:8.3f} | {best_psnr:8.3f} | {elapsed:5.1f}s{marker}"
        )

        if should_stop:
            print(f"Early stopping after {cfg['early_stop_patience']} epochs without improvement.")
            break

    best_row = max(history, key=lambda r: r["val_psnr"])
    print(f"\nBest epoch {best_row['epoch']} with val_psnr={best_row['val_psnr']:.3f} dB")
    print(f"Total time: {cumulative_time/60:.1f} min | Checkpoints: {output_dir}")
    return history

## 4. Smoke Test

Quick 2-epoch run on small subsets to verify the full pipeline (mixed data, model,
EMA, gradient clipping, combined loss) works before committing to training.

In [ ]:
print(f"Smoke test enabled: {RUN_SMOKE_TEST}")

if RUN_SMOKE_TEST:
    smoke_cfg = {**TRAIN_CFG, "epochs": 2, "early_stop_patience": 2}
    smoke_train = DataLoader(Subset(train_dataset, list(range(min(32, len(train_dataset))))), batch_size=4, shuffle=True, num_workers=0)
    smoke_train_eval = DataLoader(Subset(train_eval_dataset, list(range(min(16, len(train_eval_dataset))))), batch_size=4, shuffle=False, num_workers=0)
    smoke_val = DataLoader(Subset(val_dataset, list(range(min(16, len(val_dataset))))), batch_size=4, shuffle=False, num_workers=0)

    smoke_model = DSDANSR(**MODEL_CFG).to(device)
    smoke_opt = AdamW(smoke_model.parameters(), lr=smoke_cfg["lr"], weight_decay=smoke_cfg["weight_decay"])
    smoke_ema = EMA(smoke_model, decay=smoke_cfg["ema_decay"])

    for ep in range(smoke_cfg["epochs"]):
        tm = train_one_epoch(smoke_model, smoke_train, smoke_opt, smoke_cfg, ema=smoke_ema)
        smoke_ema.apply_shadow(smoke_model)
        te = evaluate_loader(smoke_model, smoke_train_eval, smoke_cfg, "train_eval")
        vm = evaluate_loader(smoke_model, smoke_val, smoke_cfg, "val")
        smoke_ema.restore(smoke_model)
        print(
            f"smoke epoch {ep+1}: loss={tm['train_loss']:.6f} | "
            f"online={tm['train_psnr_online']:.3f} | eval={te['train_eval_psnr']:.3f} | "
            f"val={vm['val_psnr']:.3f} dB"
        )

    del smoke_model, smoke_opt, smoke_ema
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Smoke test passed.")
else:
    print("Smoke test skipped.")

## 5. Full Training

Extended training with EMA-based validation. The `*` marker indicates new best val PSNR.
EMA weights are used for validation and final export.

In [ ]:
model = DSDANSR(**MODEL_CFG).to(device)
print(f"Target: {TRAIN_CFG['epochs']} epochs | Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Resume: {RESUME_TRAINING} | Output: {OUTPUT_DIR}\n")

history = fit(model, train_loader, train_eval_loader, val_loader, OUTPUT_DIR, cfg=TRAIN_CFG, resume=RESUME_TRAINING)

## 6. Diagnostics: Per-Source PSNR

Breakdown of model PSNR across paired and ImageNet validation sources,
plus difficulty analysis (baseline input PSNR, hardest samples).

In [ ]:
def load_checkpoint(model, checkpoint_path, map_location="cpu"):
    ckpt = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    return ckpt


@torch.no_grad()
def collect_psnr_records(model, loader, max_items=None):
    model.eval()
    records = []
    for lr_img, hr_img, names, sources in loader:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)
        pred = model(lr_img)
        pred_psnr = compute_psnr(pred, hr_img).cpu().tolist()
        input_psnr = compute_psnr(lr_img, hr_img).cpu().tolist()
        for name, source, pp, ip in zip(names, sources, pred_psnr, input_psnr):
            records.append({"name": name, "source": source, "pred_psnr": pp, "input_psnr": ip})
            if max_items and len(records) >= max_items:
                return records
    return records


def summarize_records(title, records):
    if not records:
        print(f"{title}: no records")
        return
    psnrs = torch.tensor([r["pred_psnr"] for r in records])
    baselines = torch.tensor([r["input_psnr"] for r in records])
    print(f"{title}: n={len(records)} | model={psnrs.mean():.3f} dB | baseline={baselines.mean():.3f} dB")
    print(f"  p10={torch.quantile(psnrs, 0.1):.3f} | p50={torch.quantile(psnrs, 0.5):.3f} | p90={torch.quantile(psnrs, 0.9):.3f}")
    for r in sorted(records, key=lambda r: r["pred_psnr"])[:3]:
        print(f"  hardest: {r['source']}:{r['name']} -> {r['pred_psnr']:.3f} dB")


best_ckpt = OUTPUT_DIR / "best.pt"
if RUN_DIAGNOSTICS:
    if best_ckpt.exists():
        export_cfg = {**MODEL_CFG, "dropout": 0.0, "max_drop_path": 0.0}
        diag_model = DSDANSR(**export_cfg).to(device)
        ckpt = load_checkpoint(diag_model, best_ckpt, map_location=device)
        if "ema_shadow" in ckpt:
            for name, param in diag_model.named_parameters():
                if name in ckpt["ema_shadow"]:
                    param.data.copy_(ckpt["ema_shadow"][name])
        print(f"Loaded best checkpoint from epoch {ckpt.get('epoch', '?')}")

        summarize_records("train_eval", collect_psnr_records(diag_model, train_eval_loader, DATA_CFG["train_eval_subset_size"]))
        summarize_records("paired_val", collect_psnr_records(diag_model, paired_val_loader))
        summarize_records("imagenet_val", collect_psnr_records(diag_model, imagenet_val_loader))
        summarize_records("combined_val", collect_psnr_records(diag_model, val_loader))
        del diag_model
    else:
        print(f"Run training first so {best_ckpt} exists.")
else:
    print("Diagnostics skipped.")

## 7. ONNX Export

Export best checkpoint (EMA weights) to ONNX with opset 13, fixed shape `[1, 3, 256, 256]`.
Dropout and stochastic depth are disabled for the export model. Constant folding fuses
BatchNorm into Conv.

In [ ]:
def export_to_onnx(checkpoint_path, onnx_path, verify=False, sample_loader=None):
    checkpoint_path, onnx_path = Path(checkpoint_path), Path(onnx_path)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)

    export_cfg = {**MODEL_CFG, "dropout": 0.0, "max_drop_path": 0.0}
    export_model = DSDANSR(**export_cfg).to(device)
    ckpt = load_checkpoint(export_model, checkpoint_path, map_location=device)

    if "ema_shadow" in ckpt:
        for name, param in export_model.named_parameters():
            if name in ckpt["ema_shadow"]:
                param.data.copy_(ckpt["ema_shadow"][name])
        print("Loaded EMA weights for export")

    export_model.eval()
    if "metrics" in ckpt:
        print(f"Checkpoint epoch {ckpt['metrics'].get('epoch', '?')}, val_psnr={ckpt['metrics'].get('val_psnr', '?'):.3f} dB")

    dummy = torch.randn(1, 3, DATA_CFG["eval_size"], DATA_CFG["eval_size"], device=device)
    export_kwargs = dict(export_params=True, opset_version=13, do_constant_folding=True,
                         input_names=["input"], output_names=["output"])
    try:
        torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(export_model, dummy, str(onnx_path), **export_kwargs)

    print(f"Exported to {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB)")

    if onnx is not None:
        onnx.checker.check_model(onnx.load(str(onnx_path)))
        print("ONNX checker: passed")

    if verify and ort is not None and sample_loader is not None:
        sample_lr, _, _, _ = next(iter(sample_loader))
        sample_lr = sample_lr[:1].to(device)
        with torch.no_grad():
            torch_out = export_model(sample_lr).cpu().numpy()
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_out = sess.run(["output"], {"input": sample_lr.cpu().numpy()})[0]
        diff = abs(torch_out - ort_out)
        print(f"Parity: max_diff={diff.max():.8f}, mean_diff={diff.mean():.8f}")

    return onnx_path


best_ckpt = OUTPUT_DIR / "best.pt"
if RUN_ONNX_EXPORT:
    if best_ckpt.exists():
        onnx_path = export_to_onnx(best_ckpt, OUTPUT_DIR / "best.onnx", verify=True, sample_loader=paired_val_loader)
        print(f"Ready for Mobilint compilation: {onnx_path}")
    else:
        print(f"Run training first so {best_ckpt} exists.")
else:
    print("ONNX export skipped.")

## 8. Calibration Set Export

Builds a diverse 128-sample calibration subset from training data, scored by
brightness, contrast, and texture energy for good quantization coverage.

In [ ]:
CALIBRATION_CFG = {"num_samples": 128, "seed": SEED, "output_subdir": "calibration"}


def slugify_name(text):
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in text)
    return "_".join(part for part in cleaned.split("_") if part)[:80] or "sample"


@torch.no_grad()
def score_lr_tensor(lr_t):
    gray = lr_t.float().mean(dim=0)
    brightness = float(gray.mean().item())
    contrast = float(gray.std().item())
    grad_x = gray[:, 1:] - gray[:, :-1]
    grad_y = gray[1:, :] - gray[:-1, :]
    texture = float(0.5 * (grad_x.abs().mean().item() + grad_y.abs().mean().item()))
    return {"brightness": brightness, "contrast": contrast, "texture": texture}


@torch.no_grad()
def collect_calibration_candidates(dataset_map):
    records = []
    for dk, ds in dataset_map.items():
        for di in range(len(ds)):
            lr_t, _, name, source = ds[di]
            records.append({"dataset_key": dk, "dataset_index": di, "name": name, "source": source, **score_lr_tensor(lr_t)})
    return records


def assign_tertile_bins(records, metric):
    vals = torch.tensor([r[metric] for r in records], dtype=torch.float32)
    lo, hi = torch.quantile(vals, 1/3).item(), torch.quantile(vals, 2/3).item()
    for r in records:
        r[f"{metric}_bin"] = "low" if r[metric] <= lo else ("high" if r[metric] >= hi else "mid")


def select_diverse_calibration_subset(records, num_samples, seed):
    if not records:
        return []
    tagged = [dict(r) for r in records]
    assign_tertile_bins(tagged, "brightness")
    assign_tertile_bins(tagged, "texture")
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for r in tagged:
        buckets[(r["source"], r["brightness_bin"], r["texture_bin"])].append(r)
    keys = sorted(buckets)
    for k in keys:
        rng.shuffle(buckets[k])
    target = min(num_samples, len(tagged))
    selected, seen = [], set()
    while len(selected) < target:
        progress = False
        for k in keys:
            while buckets[k]:
                r = buckets[k].pop()
                rid = (r["dataset_key"], r["dataset_index"])
                if rid in seen:
                    continue
                seen.add(rid)
                selected.append(r)
                progress = True
                break
            if len(selected) >= target:
                break
        if not progress:
            break
    return selected


def export_calibration_artifacts(records, dataset_map, output_dir, cfg):
    cal_dir = Path(output_dir) / cfg["output_subdir"]
    img_dir = cal_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    to_pil = transforms.ToPILImage()
    manifest, batch = [], []
    for idx, r in enumerate(records):
        lr_t, _, name, source = dataset_map[r["dataset_key"]][r["dataset_index"]]
        rel = f"images/{idx:03d}_{slugify_name(source)}_{slugify_name(name)}.png"
        to_pil(lr_t).save(cal_dir / rel)
        batch.append(lr_t)
        manifest.append({**r, "image_path": rel})
    torch.save(torch.stack(batch), cal_dir / "calibration_inputs.pt")
    summary = {"count": len(records)}
    (cal_dir / "manifest.json").write_text(json.dumps({"config": cfg, "summary": summary, "samples": manifest}, indent=2))
    print(f"Calibration: {len(records)} samples exported to {cal_dir}")
    return cal_dir


if RUN_CALIBRATION_EXPORT:
    cal_datasets = {"paired_train": paired_train_eval_dataset, "imagenet_train": imagenet_train_eval_dataset}
    cal_candidates = collect_calibration_candidates(cal_datasets)
    cal_selected = select_diverse_calibration_subset(cal_candidates, CALIBRATION_CFG["num_samples"], CALIBRATION_CFG["seed"])
    cal_dir = export_calibration_artifacts(cal_selected, cal_datasets, OUTPUT_DIR, CALIBRATION_CFG)
    print(f"Calibration manifest: {cal_dir / 'manifest.json'}")
else:
    print("Calibration export skipped.")